# SMC Impact Analysis

## Imports

In [ ]:
import os
from pathlib import Path
import string

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import math

## Plot style and directories

In [ ]:

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.titlesize": 14,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "lines.linewidth": 1.8
})

# Define project paths
PROJECT_DIR = Path("/mnt/hpc_acegid/home/khadmig/work/data/malaria/smc-impact-under5-malaria-africa")
ASSETS_DIR =  Path("../assets")
OUTPUT_DIR = PROJECT_DIR / "results" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Input files
MAP_CSV = PROJECT_DIR / "data_from_malaria_atlas_National_Unit_data_.csv"
SMC_TSV = ASSETS_DIR / "implementation_year.tsv"
AFRICA_SHP = ASSETS_DIR / "ne_110m_admin_0_countries.shp"

## Save figure helper

In [ ]:
def save_figure(fig, file_stem, out_dir=OUTPUT_DIR, dpi=600, bbox_inches="tight"):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    png_path = out_dir / f"{file_stem}.png"
    pdf_path = out_dir / f"{file_stem}.pdf"
    tif_path = out_dir / f"{file_stem}.tif"

    fig.savefig(png_path, dpi=dpi, bbox_inches=bbox_inches)
    fig.savefig(pdf_path, bbox_inches=bbox_inches)
    fig.savefig(tif_path, dpi=dpi, bbox_inches=bbox_inches)

    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")
    print(f"Saved: {tif_path}")

## Temporal Evolution of Malaria Burden Metrics in SMC Countries

In [ ]:
df = pd.read_csv(MAP_CSV)
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df = df[["iso3", "name", "admin_level", "metric", "units", "year", "value"]].copy()
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df = df.dropna(subset=["name", "metric", "year", "value"])
df = df[df["admin_level"] == "admin0"].copy()

smc = pd.read_csv(SMC_TSV, sep=r"\s+", engine="python")
smc.columns = smc.columns.str.strip().str.lower()

country_name_map = {
    "Burkina_Faso": "Burkina Faso",
    "Cameroon": "Cameroon",
    "Chad": "Chad",
    "Gambia": "Gambia",
    "Ghana": "Ghana",
    "Guinea": "Guinea",
    "Guinea_Bissau": "Guinea-Bissau",
    "Mali": "Mali",
    "Mauritania": "Mauritania",
    "Niger": "Niger",
    "Nigeria": "Nigeria",
    "Senegal": "Senegal",
    "Togo": "Togo",
    "Mozambique": "Mozambique",
    "Uganda": "Uganda"
}

smc["country_map"] = smc["country"].map(country_name_map)
smc["smc_start_year"] = pd.to_numeric(smc["smc_start_year"], errors="coerce")

country_order = sorted(smc["country_map"].dropna().tolist())
smc_year_lookup = smc[["country_map", "smc_start_year"]].rename(columns={"country_map": "name"})

df_smc = df[df["name"].isin(country_order)].copy()
df_smc = df_smc.merge(smc_year_lookup, on="name", how="left")

def plot_metric_panels(data, metric_name, country_order, ncols=3, figsize=(16, 18)):
    metric_df = data[data["metric"] == metric_name].copy()
    n = len(country_order)
    nrows = int(np.ceil(n / ncols))

    y_max = metric_df["value"].max()
    y_min = metric_df["value"].min()

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, sharex=True, sharey=True)
    axes = np.array(axes).reshape(-1)
    letters = list(string.ascii_uppercase)

    for i, country in enumerate(country_order):
        ax = axes[i]
        sub = metric_df[metric_df["name"] == country].sort_values("year").copy()

        ax.plot(sub["year"], sub["value"])

        smc_year = sub["smc_start_year"].dropna().iloc[0] if sub["smc_start_year"].notna().any() else None
        if smc_year is not None:
            ax.axvline(smc_year, linestyle="--", linewidth=1.2, color="red")

        ax.set_ylim(y_min, y_max)
        ax.set_title(country, loc="left", fontweight="bold")
        ax.set_xlabel("Year", fontweight="bold")
        ax.set_ylabel(metric_name, fontweight="bold")
        ax.text(
            0.00, 1.08, letters[i],
            transform=ax.transAxes,
            fontsize=12,
            fontweight="bold",
            va="bottom",
            ha="left"
        )

        for label in ax.get_xticklabels():
            label.set_fontweight("bold")
        for label in ax.get_yticklabels():
            label.set_fontweight("bold")

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    legend_elements = [
        Line2D([0], [0], linewidth=1.8, label=metric_name),
        Line2D([0], [0], linestyle="--", linewidth=1.2, color="red", label="SMC start year")
    ]

    fig.suptitle(
        f"Temporal evolution of {metric_name} in SMC countries",
        y=1.00,
        fontweight="bold"
    )

    fig.legend(
        handles=legend_elements,
        loc="upper center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.5, 0.98),
        prop={"weight": "bold"}
    )

    fig.tight_layout(rect=[0, 0, 1, 0.96])

    return fig, axes

fig1, _ = plot_metric_panels(df_smc, "Incidence Rate", country_order, ncols=3, figsize=(16, 18))
save_figure(fig1, "figure1_incidence_rate_panels_smc_countries")
plt.show()

fig2, _ = plot_metric_panels(df_smc, "Infection Prevalence", country_order, ncols=3, figsize=(16, 18))
save_figure(fig2, "figure1_infection_prevalence_panels_smc_countries")
plt.show()

fig3, _ = plot_metric_panels(df_smc, "Mortality Rate", country_order, ncols=3, figsize=(16, 18))
save_figure(fig3, "figure1_mortality_rate_panels_smc_countries")
plt.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D

world = gpd.read_file(AFRICA_SHP)
africa = world[world["CONTINENT"] == "Africa"].copy()

map_name_fix = {
    "Dem. Rep. Congo": "Democratic Republic of the Congo",
    "Central African Rep.": "Central African Republic",
    "Eq. Guinea": "Equatorial Guinea",
    "eSwatini": "Eswatini",
    "W. Sahara": "Western Sahara",
    "S. Sudan": "South Sudan",
    "Côte d'Ivoire": "Ivory Coast"
}

africa["ADMIN"] = africa["ADMIN"].replace(map_name_fix)

metric_title_map = {
    "Incidence Rate": "Incidence Rate",
    "Infection Prevalence": "Infection Prevalence",
    "Mortality Rate": "Mortality Rate"
}

country_code_map = {
    "Senegal": "SN",
    "Gambia": "GM",
    "Guinea-Bissau": "GW",
    "Guinea": "GN",
    "Mauritania": "MR",
    "Mali": "ML",
    "Burkina Faso": "BF",
    "Ghana": "GH",
    "Togo": "TG",
    "Niger": "NE",
    "Chad": "TD",
    "Nigeria": "NG",
    "Cameroon": "CM",
    "Uganda": "UG",
    "Mozambique": "MZ"
}

country_positions = {
    "Senegal": (-20.0, 23.0),
    "Gambia": (-20.0, 15.0),
    "Guinea-Bissau": (-20.0, 6.0),
    "Guinea": (-12.0, -1.5),
    "Mauritania": (-10.8, 30.0),
    "Mali": (5.0, 30.0),
    "Burkina Faso": (15.0, 30.0),
    "Ghana": (-3.0, -1.5),
    "Togo": (7.0, -1.5),
    "Niger": (29.0, 29.0),
    "Chad": (29.8, 15.2),
    "Nigeria": (16.0, -1.5),
    "Cameroon": (23.5, 4.0),
    "Uganda": (43.0, 5.0),
    "Mozambique": (42.0, -11.5)
}

end_arrow_point = {
    "Senegal": (-24, 19),
    "Gambia": (-24, 14),
    "Guinea-Bissau": (-24, 5),
    "Guinea": (-20, 2),
    "Mauritania": (-15.8, 27.5),
    "Mali": (0, 27.5),
    "Burkina Faso": (15.0, 27.5),
    "Ghana": (-10.0, 0.8),
    "Togo": (2.0, 0.8),
    "Niger": (23.0, 26.5),
    "Chad": (26.3, 15.2),
    "Nigeria": (11.0, 0.8),
    "Cameroon": (17, 5),
    "Uganda": (39.5, 5.0),
    "Mozambique": (38.5, -11.5)
}

def prepare_metric_series(data, metric_name):
    d = data[data["metric"] == metric_name].copy()
    d = d[d["name"].isin(country_code_map.keys())].copy()
    d["period_color"] = np.where(d["year"] < d["smc_start_year"], "royalblue", "crimson")
    return d

def build_centroids(gdf):
    c = gdf.copy()
    c["cx"] = c.geometry.representative_point().x
    c["cy"] = c.geometry.representative_point().y
    return c[["ADMIN", "cx", "cy"]]

def add_yearly_bar_inset(ax, sub, x_center, y_center, country_code, width_deg=7.0, height_deg=4.6, y_max=None):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    x_frac = (x_center - xmin) / (xmax - xmin)
    y_frac = (y_center - ymin) / (ymax - ymin)
    w_frac = width_deg / (xmax - xmin)
    h_frac = height_deg / (ymax - ymin)

    inset = ax.inset_axes([x_frac - w_frac / 2, y_frac - h_frac / 2, w_frac, h_frac])

    years = sub["year"].astype(int).tolist()
    values = sub["value"].tolist()
    colors = sub["period_color"].tolist()

    inset.bar(years, values, color=colors, width=0.82, linewidth=0)
    inset.set_xlim(min(years) - 0.6, max(years) + 0.6)

    if y_max is not None:
        inset.set_ylim(0, y_max)

    xticks = [2000, 2005, 2010, 2015, 2020, 2024]
    xticks = [x for x in xticks if min(years) <= x <= max(years)]
    inset.set_xticks(xticks)
    inset.set_xticklabels([str(x) for x in xticks], fontsize=4.0, fontweight="bold", rotation=90)

    inset.tick_params(axis="y", labelsize=4.0, width=0.5, length=1.5, pad=1)
    inset.tick_params(axis="x", width=0.5, length=1.5, pad=1)

    for label in inset.get_yticklabels():
        label.set_fontweight("bold")

    for spine in inset.spines.values():
        spine.set_linewidth(0.6)
        spine.set_color("black")

    inset.set_facecolor("white")
    inset.text(
        0.02, 0.98, country_code,
        transform=inset.transAxes,
        ha="left",
        va="top",
        fontsize=7,
        fontweight="bold",
        color="black"
    )
    return inset

def draw_arrow_to_inset(ax, x0, y0, x_end, y_end):
    arrow = FancyArrowPatch(
        (x0, y0),
        (x_end, y_end),
        arrowstyle="-|>",
        mutation_scale=10,
        linewidth=1.0,
        color="black",
        shrinkA=2,
        shrinkB=0,
        connectionstyle="arc3,rad=0.0",
        zorder=3
    )
    ax.add_patch(arrow)

def plot_africa_metric_map_with_yearly_bars(ax, data, metric_name, africa_gdf, panel_letter):
    d = prepare_metric_series(data, metric_name)
    smc_countries = sorted(d["name"].dropna().unique().tolist())

    africa_plot = africa_gdf.copy()
    africa_plot["is_smc"] = africa_plot["ADMIN"].isin(smc_countries)

    centroids = build_centroids(africa_plot[africa_plot["is_smc"]].copy())
    africa_plot = africa_plot.merge(centroids, on="ADMIN", how="left")

    y_max = d["value"].max() * 1.08
    box_width = 7.0
    box_height = 4.6

    africa_plot[~africa_plot["is_smc"]].plot(
        ax=ax,
        color="#f2f2f2",
        edgecolor="#c9c9c9",
        linewidth=0.8,
        zorder=1
    )

    africa_plot[africa_plot["is_smc"]].plot(
        ax=ax,
        color="#d9ecff",
        edgecolor="black",
        linewidth=1.4,
        zorder=2
    )

    for _, row in africa_plot[africa_plot["is_smc"]].iterrows():
        ax.text(
            row["cx"],
            row["cy"],
            row["ADMIN"],
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold",
            color="black",
            zorder=5
        )

    for country in smc_countries:
        sub = d[d["name"] == country].sort_values("year").copy()
        row = africa_plot[africa_plot["ADMIN"] == country].iloc[0]

        x0, y0 = row["cx"], row["cy"]
        x1, y1 = country_positions[country]
        xe, ye = end_arrow_point[country]
        code = country_code_map[country]

        draw_arrow_to_inset(ax, x0, y0, xe, ye)

        add_yearly_bar_inset(
            ax=ax,
            sub=sub,
            x_center=x1,
            y_center=y1,
            country_code=code,
            width_deg=box_width,
            height_deg=box_height,
            y_max=y_max
        )

    ax.set_title(metric_title_map[metric_name], fontsize=18, fontweight="bold", pad=10)
    ax.set_xlim(-30, 56)
    ax.set_ylim(-36, 38)
    ax.set_axis_off()

    ax.text(
        0.0, 1.02, panel_letter,
        transform=ax.transAxes,
        fontsize=18,
        fontweight="bold",
        va="bottom",
        ha="left"
    )

fig, axes = plt.subplots(2, 2, figsize=(28, 24))

plot_africa_metric_map_with_yearly_bars(axes[0, 0], df_smc, "Incidence Rate", africa, "A")
plot_africa_metric_map_with_yearly_bars(axes[0, 1], df_smc, "Infection Prevalence", africa, "B")
plot_africa_metric_map_with_yearly_bars(axes[1, 0], df_smc, "Mortality Rate", africa, "C")
axes[1, 1].axis("off")

legend_handles = [
    Line2D([0], [0], color="royalblue", linewidth=8, label="Before SMC year"),
    Line2D([0], [0], color="crimson", linewidth=8, label="SMC year and after"),
    Line2D([0], [0], color="#d9ecff", linewidth=8, label="SMC country"),
    Line2D([0], [0], color="black", linewidth=1.2, label="Arrow to country")
]

leg = fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.03),
    ncol=4,
    frameon=True,
    fontsize=12,
    prop={"weight": "bold"}
)
leg.get_frame().set_edgecolor("black")
leg.get_frame().set_linewidth(0.8)
leg.get_frame().set_alpha(1)

fig.suptitle(
    "Africa Map with Annual Malaria Trajectories in SMC Countries",
    fontsize=24,
    fontweight="bold",
    y=0.985
)

fig.tight_layout(rect=[0, 0.06, 1, 0.97])
save_figure(fig, "figure2_panel_2x2_africa_map_incidence_prevalence_mortality_smc_countries")
plt.show()

## Temporal and Geographic Dynamics of Sulfadoxine–Pyrimethamine and Amodiaquine Resistance Across SMC Countries

### Heatmap (SP & AQ Haplotypes)

In [ ]:
haplo = pd.read_csv(
    "/mnt/hpc_acegid/home/khadmig/work/data/malaria/smc-impact-under5-malaria-africa/results/genomics_smc_malaGen/pf8_smc_15countries_haplotype_prevalence_for_plot.tsv",
    sep="\t"
)

haplo = haplo.dropna(subset=["country", "year", "haplotype", "prevalence"]).copy()
haplo["year"] = haplo["year"].astype(int)

sp_haplotypes = ["dhfr_triple", "sp_quadruple", "sp_quintuple", "sp_sextuple", "sp_sextuple_dhfr164"]
aq_haplotypes = ["mdr1_86Y", "mdr1_184F", "mdr1_double", "pfcrt_76T", "pfcrt_triple", "aq_resistant_profile"]

haplotype_mutations = {
    "dhfr_triple": "N51I + C59R + S108N",
    "sp_quadruple": "A437G + N51I + C59R + S108N",
    "sp_quintuple": "A437G + K540E + N51I + C59R + S108N",
    "sp_sextuple": "A437G + K540E + A581G + N51I + C59R + S108N",
    "sp_sextuple_dhfr164": "A437G + K540E + N51I + C59R + S108N + I164L",
    "mdr1_86Y": "N86Y",
    "mdr1_184F": "Y184F",
    "mdr1_double": "N86Y + Y184F",
    "pfcrt_76T": "K76T",
    "pfcrt_triple": "M74I + N75E + K76T",
    "aq_resistant_profile": "N86Y + Y184F + K76T",
}

countries = sorted(haplo["country"].unique())
years = sorted(haplo["year"].unique())

def plot_haplotype_heatmap_panel(haplotype_list, figure_title, output_name):
    ncols = 2
    nrows = math.ceil(len(haplotype_list) / ncols)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(22, 3.8 * nrows),
        sharex=True,
        sharey=True
    )

    axes = np.array(axes).reshape(-1)
    letters = list(string.ascii_uppercase)
    im = None

    for i, hap in enumerate(haplotype_list):
        ax = axes[i]

        sub = haplo[haplo["haplotype"] == hap].copy()

        pivot = sub.pivot_table(
            index="country",
            columns="year",
            values="prevalence",
            aggfunc="first"
        ).reindex(index=countries, columns=years)

        im = ax.imshow(
            pivot.values,
            aspect="auto",
            vmin=0,
            vmax=1,
            cmap="inferno"
        )

        ax.set_yticks(range(len(countries)))
        ax.set_yticklabels(countries, fontsize=9, fontweight="bold")

        ax.set_xticks(range(len(years)))
        if i >= len(haplotype_list) - ncols:
            ax.set_xticklabels(years, rotation=90, fontsize=8, fontweight="bold")
            ax.set_xlabel("Year", fontsize=11, fontweight="bold")
        else:
            ax.set_xticklabels([])

        ax.set_title(
            f"{hap} ({haplotype_mutations[hap]})",
            fontsize=13,
            fontweight="bold",
            loc="left"
        )

        ax.text(
            -0.12, 1.05, letters[i],
            transform=ax.transAxes,
            fontsize=14,
            fontweight="bold",
            va="bottom",
            ha="left"
        )

        if i % ncols == 0:
            ax.set_ylabel("Country", fontsize=11, fontweight="bold")

    for j in range(len(haplotype_list), len(axes)):
        axes[j].set_visible(False)

    fig.subplots_adjust(right=0.90, top=0.92, hspace=0.35, wspace=0.12)

    cbar_ax = fig.add_axes([0.92, 0.25, 0.015, 0.50])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label("Prevalence", fontsize=12, fontweight="bold")
    for tick in cbar.ax.get_yticklabels():
        tick.set_fontweight("bold")

    fig.suptitle(
        figure_title,
        fontsize=22,
        fontweight="bold",
        y=0.98
    )

    save_figure(fig, output_name)
    plt.show()

plot_haplotype_heatmap_panel(
    sp_haplotypes,
    "Temporal Landscape of SP Resistance Haplotypes",
    "figure3_1_panel_heatmap_sp_haplotypes_by_country_year"
)

plot_haplotype_heatmap_panel(
    aq_haplotypes,
    "Temporal Landscape of AQ Resistance Haplotypes",
    "figure3_2_panel_heatmap_aq_haplotypes_by_country_year"
)

### Evolution (SP & AQ Haplotypes)

In [ ]:
haplo = pd.read_csv(
    "/mnt/hpc_acegid/home/khadmig/work/data/malaria/smc-impact-under5-malaria-africa/results/genomics_smc_malaGen/pf8_smc_15countries_haplotype_prevalence_for_plot.tsv",
    sep="\t"
)

haplo = haplo.dropna(subset=["country", "year", "haplotype", "prevalence"]).copy()
haplo["year"] = haplo["year"].astype(int)

sp_haplotypes = ["dhfr_triple", "sp_quadruple", "sp_quintuple", "sp_sextuple", "sp_sextuple_dhfr164"]
aq_haplotypes = ["mdr1_86Y", "mdr1_184F", "mdr1_double", "pfcrt_76T", "pfcrt_triple", "aq_resistant_profile"]

haplotype_mutations = {
    "dhfr_triple": "N51I + C59R + S108N",
    "sp_quadruple": "A437G + N51I + C59R + S108N",
    "sp_quintuple": "A437G + K540E + N51I + C59R + S108N",
    "sp_sextuple": "A437G + K540E + A581G + N51I + C59R + S108N",
    "sp_sextuple_dhfr164": "A437G + K540E + N51I + C59R + S108N + I164L",
    "mdr1_86Y": "N86Y",
    "mdr1_184F": "Y184F",
    "mdr1_double": "N86Y + Y184F",
    "pfcrt_76T": "K76T",
    "pfcrt_triple": "M74I + N75E + K76T",
    "aq_resistant_profile": "N86Y + Y184F + K76T",
}

countries = sorted(haplo["country"].unique())
years = sorted(haplo["year"].unique())

country_colors = {
    "Burkina Faso": "red",
    "Cameroon": "blue",
    "Chad": "darkorange",
    "Gambia": "green",
    "Ghana": "orange",
    "Guinea": "purple",
    "Guinea-Bissau": "teal",
    "Mali": "cyan",
    "Mauritania": "magenta",
    "Mozambique": "gold",
    "Niger": "darkred",
    "Nigeria": "black",
    "Senegal": "brown",
    "Togo": "darkgreen",
    "Uganda": "lime",
}

def make_panel_figure(haplotype_list, figure_title, output_name):
    ncols = 2
    nrows = math.ceil(len(haplotype_list) / ncols)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(24, 4.2 * nrows),
        sharex=True,
        sharey=True
    )

    axes = np.array(axes).reshape(-1)
    letters = list(string.ascii_uppercase)

    for i, hap in enumerate(haplotype_list):
        ax = axes[i]
        sub = haplo[haplo["haplotype"] == hap].copy()

        for country in countries:
            country_sub = sub[sub["country"] == country].sort_values("year")
            if country_sub.empty:
                continue

            ax.plot(
                country_sub["year"],
                country_sub["prevalence"],
                linewidth=2.2,
                marker="o",
                markersize=4,
                label=country,
                color=country_colors.get(country, None)
            )

        ax.set_ylim(0, 1)
        ax.set_xticks(years)

        if i >= len(haplotype_list) - ncols:
            ax.set_xticklabels(years, rotation=90, fontsize=8, fontweight="bold")
            ax.set_xlabel("Year", fontsize=11, fontweight="bold")
        else:
            ax.set_xticklabels([])

        ax.set_title(
            f"{hap} ({haplotype_mutations[hap]})",
            fontsize=13,
            fontweight="bold",
            loc="left"
        )

        ax.text(
            -0.12, 1.05, letters[i],
            transform=ax.transAxes,
            fontsize=14,
            fontweight="bold",
            va="bottom",
            ha="left"
        )

        if i % ncols == 0:
            ax.set_ylabel("Prevalence", fontsize=11, fontweight="bold")

        for label in ax.get_xticklabels():
            label.set_fontweight("bold")
        for label in ax.get_yticklabels():
            label.set_fontweight("bold")

    for j in range(len(haplotype_list), len(axes)):
        axes[j].set_visible(False)

    legend_handles = [
        Line2D([0], [0], color=country_colors.get(country, "gray"), lw=2.5, marker="o", markersize=5, label=country)
        for country in countries if country in haplo["country"].unique()
    ]

    fig.subplots_adjust(right=0.83, top=0.93, hspace=0.35, wspace=0.14)

    fig.legend(
        handles=legend_handles,
        loc="center left",
        bbox_to_anchor=(0.845, 0.5),
        frameon=False,
        prop={"weight": "bold", "size": 10},
        title="Country",
        title_fontproperties={"weight": "bold", "size": 11}
    )

    fig.suptitle(
        figure_title,
        fontsize=22,
        fontweight="bold",
        y=0.995
    )

    save_figure(fig, output_name)
    plt.show()

make_panel_figure(
    sp_haplotypes,
    "Temporal Landscape of SP Resistance Haplotypes",
    "figure3_1_panel_lineplot_sp_haplotypes_by_country_year"
)

make_panel_figure(
    aq_haplotypes,
    "Temporal Landscape of AQ Resistance Haplotypes",
    "figure3_2_panel_lineplot_aq_haplotypes_by_country_year"
)

## Our Data Analysis